In [13]:
import argparse
import os

import anndata as ad

from methyltrain.api.prepare import prepare_cohort
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import CohortLayout
from methyltrain.api.steps import save_cohort

from typing import Dict, Tuple

import anndata as ad
import pandas as pd

from methyltrain.fs.layout import ProjectLayout, CohortLayout
from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    filter_by_mad,
    cohort_batch_correction,
    aggregate_genes,
    clip_and_scale,
    split,
    load_raw_project,
    load_processed_project,
    save_cohort
)

verbose = True
config = "/ddn_exa/campbell/sli/methyltrain/config/pancancer_config.yaml"

In [14]:
# Load the user-provided configurations
config = load_config(config)
cohort = config.get('project_id', '')

# Initialize the projects provided in the configurations
project_dir = config.get('project_dir', '')
all_projects = config.get('projects', [])
project_list = []
for project in all_projects:
    project_list.append(os.path.join(project_dir, f"{project}_adata.h5ad"))

# Initialize the default cohort layout
layout = CohortLayout(
    cohort_name = cohort,
    root_dir = "./data",
    project_list = project_list,
    cohort_adata = f"../data/cohort/{cohort}_cohort_adata.h5ad",
    train_adata = f"../data/cohort/{cohort}_train_adata.h5ad",
    val_adata = f"../data/cohort/{cohort}_val_adata.h5ad",
    test_adata = f"../data/cohort/{cohort}_test_adata.h5ad"
)
layout.initialize()
layout.validate()

In [12]:
# Load each processed project AnnData object 
if verbose: print("Attempting to load all project AnnData objects")
project_adatas = [load_processed_project(path) 
                  for path in layout.project_list]
if verbose: print("Successfully loaded all project AnnData objects")

Attempting to load all project AnnData objects
Successfully loaded all project AnnData objects


In [14]:
# Aggregate the projects into a cohort AnnData object
if verbose: print("Attempting to aggregate the cohort")
cohort_adata = aggregate_cohort(project_adatas, layout)
if verbose: print("Successfully aggregated the cohort")

# Perform MAD probe filtering to reduce probe dimensionality
if config.get('toggles', {}).get('MAD_probe_filtering', True):
    if verbose: print("Attempting to perform MAD probe filtering")
    cohort_adata = filter_by_mad(cohort_adata, config)
    if verbose: print("Successfully performed MAD probe filtering")

# Perform batch effect correction across datasets if toggled
if config.get('toggles', {}).get('batch_correction', True):
    if verbose: print("Attempting to perform batch correction")
    cohort_adata = cohort_batch_correction(cohort_adata, config)
    if verbose: print("Successfully performed batch correction")

# Optionally aggregate to the gene-level if toggled by user configurations
if config.get('toggles', {}).get('gene_aggregation', True):
    if verbose: print("Attempting to perform gene aggregation")
    cohort_adata = aggregate_genes(cohort_adata, config)
    if verbose: print("Successfully aggregated to the gene-level")

# Winsorize and scale M-values to [-1, 1] promote downstream ML stability
if config.get('toggles', {}).get('clip_and_scale', True):
    if verbose: print("Attempting to perform winsorization and Min-max scaling")
    cohort_adata = clip_and_scale(cohort_adata, config)
    if verbose: print("Successfully winsorized and scaled the cohort")

# Split the cohort AnnData object into train-val-test splits
if config.get('toggles', {}).get('split', True):
    if verbose: print("Attempting to split the cohort into training sets")
    train_adata, val_adata, test_adata = split(cohort_adata, config)
    if verbose: print("Successfully split the cohort into training sets")
else:
    if verbose: print("Did not split the cohort into training sets (not toggled)")
    train_adata, val_adata, test_adata = None, None, None

# Save the processed cohort adata prior to splitting
if verbose: print("Attempting to save the cohort AnnData object")
save_cohort(cohort_adata, layout)
if verbose: print("Successfully saved the cohort AnnData object")

Attempting to aggregate the cohort
Successfully aggregated the cohort
Did not split the cohort into training sets (not toggled)
Attempting to save the cohort AnnData object
Successfully saved the cohort AnnData object


In [4]:
adata = ad.read_h5ad("/ddn_exa/campbell/sli/methyltrain/data/cohort/pancancer_cohort_adata.h5ad")
adata.obs

,file_name,data_type,data_category,experimental_strategy,platform,project_id,submitter_id,sample_type,aliquot_id,status,barcode,missing_rate,mean
file_id,,,,,,,,,,,,,
fce94fc9-add0-475f-b5ec-e756135a3800,00a7b514-c362-4827-902c-a85f64fad069.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-BLCA,TCGA-GD-A6C6,Primary Tumor,58ccde98-246d-4c8a-b9d0-84c446350d62,success,TCGA-GD-A6C6-01A-21D-A31M-05,0.177077,0.494169
6ecba7dc-d639-4b1a-887d-1653835836f3,028a443c-d111-4a35-b144-040ea07a371d.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-BLCA,TCGA-5N-A9KI,Primary Tumor,74c55498-f8a4-4cef-8568-e687108dc75d,success,TCGA-5N-A9KI-01A-31D-A42F-05,0.156400,0.378228
bd5ee663-48c7-4394-82f9-395a50c102ee,02d1fb77-4a36-4ee8-ab49-2daee65970d2.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-BLCA,TCGA-DK-A3IV,Primary Tumor,00d778c1-72c0-40d6-893e-c2c2ce79263d,success,TCGA-DK-A3IV-01A-22D-A21B-05,0.151515,0.468837
cb3f0af0-320d-49bc-87db-3db0b4c24c84,02e041c2-4e3d-42d1-b16f-d345d043b2a7.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-BLCA,TCGA-FD-A6TA,Primary Tumor,1e5a8a8d-0155-4716-8c1f-777eac5283e1,success,TCGA-FD-A6TA-01A-12D-A33I-05,0.149190,0.415871
84cfc1b8-3077-4592-8a79-d40b7b53ce45,0413ea7a-f95b-43e8-9814-523189c7febc.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-BLCA,TCGA-UY-A78O,Primary Tumor,89f8e3ab-75c9-4586-8dd4-bf9ef62af7d4,success,TCGA-UY-A78O-01A-12D-A33I-05,0.159811,0.468969
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6d3798ab-7efe-4ef9-bc1f-704b46565183,fb0e9757-53f6-4cc1-b649-0cc942e2fb74.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UCEC,TCGA-AJ-A5DW,Primary Tumor,54f2c9b1-b21c-4ab5-8cf2-6877caf7b9f7,success,TCGA-AJ-A5DW-01A-11D-A27Y-05,0.141051,0.500613
66962fd4-cbe4-4de5-af5d-60dec04efcfc,fcda19c9-a0bb-42eb-8fd5-d9aca2cc09f0.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UCEC,TCGA-EO-A2CG,Primary Tumor,767da78b-b164-4de5-93fb-78cc1ae198a5,success,TCGA-EO-A2CG-01A-12D-A17Z-05,0.142219,0.488044
a3f99f64-f731-46bd-b81d-691790b391b0,fe8b5387-c97f-4384-90ac-1b4ae3c1786a.methylati...,Methylation Beta Value,DNA Methylation,Methylation Array,Illumina Human Methylation 450,TCGA-UCEC,TCGA-BG-A0MH,Primary Tumor,e2a0e977-5ec1-4158-b1bb-c0813ef635d5,success,TCGA-BG-A0MH-01A-11D-A123-05,0.135887,0.457883


In [7]:
cohort_adata = adata

In [10]:
train_adata, val_adata, test_adata = split(cohort_adata, config)

In [15]:
# Save the training splits if toggled
train_adata.write_h5ad(layout.train_adata)
val_adata.write_h5ad(layout.val_adata)
test_adata.write_h5ad(layout.test_adata)